# PagedAttention under pressure: forcing vLLM's block manager to preempt, and watching it

Qwen2.5-1.5B-Instruct on a T4. The KV cache is shrunk with `--gpu-memory-utilization` so that an open-loop load with long outputs overflows it; `/metrics` is sampled every 0.5 s (KV usage, running, waiting, preemptions, tokens). Configurations: healthy baseline; pressure; pressure with block size 32; pressure with prefix caching off. ~10 min install, ~4 min per configuration.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/pagedattention-under-pressure" "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, paged_pressure.timeline, servellm.launch, tokenbench
from tokenbench.load import WATCHED; assert "preemptions" in WATCHED, "tokenbench too old"
assert torch.cuda.is_available()
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__)

## Configurations

The load is the same everywhere: open loop, 6 req/s, 512-token prompts with 384 output tokens forced, 90 s. With the full cache this is comfortable; with utilisation 0.35 it is not.

In [ ]:
import json, subprocess, time
from servellm.launch import ServerConfig, VllmServer
from paged_pressure.timeline import server_args, summarize, preemption_events, annotate

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"; BASE = "http://127.0.0.1:8000"
CONFIGS = [
    ("healthy (util 0.9, block 16, prefix on)", 0.9, 16, True),
    ("pressure (util 0.35, block 16, prefix on)", 0.35, 16, True),
    ("pressure, block 32", 0.35, 32, True),
    ("pressure, prefix caching off", 0.35, 16, False),
]
runs = []
for name, util, bs, pc in CONFIGS:
    tag = name.split(" (")[0].replace(" ", "_").replace(",", "")
    server = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=2048, gpu_memory_utilization=util,
                                     extra_args=tuple(server_args(util, bs, pc, 256))), log_path=f"results/startup_{tag}.log")
    rec = {"config": name, "utilization": util, "block_size": bs, "prefix_caching": pc}
    try:
        info = server.start(ready_timeout_s=1200)
        rec["kv_cache_tokens"], rec["kv_cache_gib"] = info.kv_cache_tokens, info.kv_cache_gib
        subprocess.run(["tokenbench", "sweep", "--base-url", BASE, "--model", MODEL, "--out", "results/_warmup.json", "--levels", "1,8",
                        "--num-requests", "16", "--warmup", "0", "--prompt-tokens", "64", "--max-tokens", "64", "--request-timeout", "60"], check=True, capture_output=True, timeout=300)
        subprocess.run(["tokenbench", "sweep", "--base-url", BASE, "--model", MODEL, "--out", f"results/{tag}.json", "--mode", "open",
                        "--levels", "6", "--duration", "90", "--warmup", "0", "--prompt-tokens", "512", "--max-tokens", "384",
                        "--metrics-interval", "0.5", "--request-timeout", "120"], check=True, capture_output=True, timeout=600)
        lv = json.load(open(f"results/{tag}.json"))["levels"][0]
        rec["level"] = {k: lv[k] for k in ("requests", "errors", "output_tokens_per_s", "ttft", "itl", "e2e")}
        rec["timeline"] = lv["metrics_timeline"]
        rec["summary"] = summarize(rec["timeline"]); rec["events"] = preemption_events(rec["timeline"])
        s = rec["summary"]
        print(f"{name:40s} cache {info.kv_cache_tokens} tok | preemptions {s['preemptions_total']} | full {s['time_at_full_cache_pct']:.0f}% of time | "
              f"peak waiting {s['peak_waiting']} | {s['mean_tokens_per_s']:.0f} tok/s | ITL p99 {lv['itl']['p99']*1000:.0f} ms | TTFT p50 {lv['ttft']['p50']*1000:.0f} ms")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"; print(name, "FAILED:", rec["error"])
    finally:
        server.stop(); time.sleep(5)
    runs.append(rec)
    json.dump(runs, open("results/runs.json", "w"), indent=1)

## Timelines

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
ok = [r for r in runs if "error" not in r]
fig, axes = plt.subplots(len(ok), 1, figsize=(12, 3.2 * len(ok)), sharex=True)
axes = [axes] if len(ok) == 1 else list(axes)
for ax, r in zip(axes, ok):
    tl = annotate(r["timeline"]); t = [s["t"] for s in tl]
    ax.plot(t, [100 * (s.get("kv_cache_usage") or 0) for s in tl], color="tab:red", label="KV cache usage %")
    ax2 = ax.twinx()
    ax2.plot(t, [s.get("running") or 0 for s in tl], color="tab:blue", label="running")
    ax2.plot(t, [s.get("waiting") or 0 for s in tl], color="tab:orange", label="waiting")
    for e in r["events"]:
        ax.axvline(e["t"], color="k", alpha=0.25, lw=0.8)
    ax.set_ylabel("cache %"); ax2.set_ylabel("requests"); ax.set_ylim(0, 105)
    ax.set_title(f"{r['config']}: cache {r['kv_cache_tokens']:,} tokens, {r['summary']['preemptions_total']} preemptions (vertical lines), {r['summary']['mean_tokens_per_s']:.0f} tok/s")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right"); ax.grid(alpha=.3)
axes[-1].set_xlabel("time (s)")
fig.tight_layout(); fig.savefig("results/timelines.png", dpi=120)
from IPython.display import Image, display; display(Image("results/timelines.png"))

In [ ]:
rows = []
for r in ok:
    s, lv = r["summary"], r["level"]
    rows.append({"config": r["config"], "kv_tokens": r["kv_cache_tokens"], "preemptions": s["preemptions_total"], "events": s["preemption_events"],
                 "full_pct": round(s["time_at_full_cache_pct"]), "peak_waiting": s["peak_waiting"], "tok_s": round(s["mean_tokens_per_s"]),
                 "tok_s_steady": round(s["tokens_per_s_steady"] or 0), "tok_s_after_preempt": round(s["tokens_per_s_after_preemption"] or 0),
                 "ttft_p50_ms": round(lv["ttft"]["p50"] * 1000), "itl_p50_ms": round(lv["itl"]["p50"] * 1000, 1), "itl_p99_ms": round(lv["itl"]["p99"] * 1000),
                 "e2e_p95_s": round(lv["e2e"]["p95"], 1), "requests": lv["requests"], "errors": lv["errors"]})
head = list(rows[0].keys())
md_ = "| " + " | ".join(head) + " |\n|" + "---|" * len(head) + "\n" + "\n".join("| " + " | ".join(str(r[h]) for h in head) + " |" for r in rows) + "\n"
open("results/comparison.md", "w").write(md_); json.dump(rows, open("results/comparison.json", "w"), indent=1)
from IPython.display import Markdown; display(Markdown(md_))
for r in runs:
    if "error" in r: print("FAILED:", r["config"], r["error"][:300])

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip